# Part 3 — Multi-head attention & positional encoding

⏱ **~2 hours** · 📁 you write code in **`minitf/attention.py`** (`MultiHeadAttention`) and **`minitf/positional.py`** · 📄 paper §3.2.2, §3.5

**After this notebook you can explain:**
- why attention needs **learned projections** $W^Q, W^K, W^V$
- why we use **several heads**, and why that costs no more than one big head
- exactly how tensors are reshaped inside multi-head attention
- that attention on its own **ignores word order**, and how sinusoidal positional encodings fix that
- learned vs. sinusoidal vs. RoPE positions

In [ ]:
# Setup: run this cell first.
%load_ext autoreload
%autoreload 2
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "minitf").is_dir() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from minitf import check, viz
from minitf.attention import scaled_dot_product_attention, MultiHeadAttention
from minitf.positional import sinusoidal_positional_encoding
torch.manual_seed(0);

## 1. Why learned projections?

What if we used the token vectors directly as queries, keys **and** values: $Q = K = V = X$?

In [ ]:
x = torch.randn(6, 32)                        # 6 tokens, 32 numbers each
_, w = scaled_dot_product_attention(x, x, x)
viz.plot_attention(w, title="Q = K = V = X: every token mostly looks at itself")

A vector always matches itself best, since $x\cdot x = \|x\|^2$ is usually the largest dot product. So every token mostly attends to **itself**, and the matrix is symmetric. That's useless.

With learned projections $q = xW^Q$, $k = xW^K$, $v = xW^V$, the model learns **separately**:
- $W^Q$: *what should this token look for?* (a pronoun might look for a noun earlier in the sentence)
- $W^K$: *how should this token advertise itself?*
- $W^V$: *what information should it pass on when it's chosen?*

## 2. Why multiple heads?

One softmax computes **one** weighted average, so it captures **one kind of relationship** at a time. A token often needs several at once. In *"The **animal** didn't cross the street because **it** was tired"*, "it" might need one head for *what does "it" refer to?* and another for *what's the verb near me?*

**Multi-head attention** runs $h$ attentions in parallel, each with its own **smaller** $W^Q_i, W^K_i, W^V_i$, then concatenates the results and mixes them with $W^O$:

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, ..., \text{head}_h)\,W^O, \qquad \text{head}_i = \text{Attention}(XW^Q_i, XW^K_i, XW^V_i)$$

**The trick (it isn't $h$ times more expensive):** use one big `nn.Linear(d_model, d_model)` for all the heads' queries together, then **split** the result into $h$ heads of $d_k = d_{model}/h$ numbers each. You wrote `split_heads` in Part 0!

```
x_q  (B, T_q, d_model) ─ w_q ─► (B, T_q, d_model) ─ split ─► q (B, h, T_q, d_k)
x_kv (B, T_k, d_model) ─ w_k ─► (B, T_k, d_model) ─ split ─► k (B, h, T_k, d_k)
x_kv (B, T_k, d_model) ─ w_v ─► (B, T_k, d_model) ─ split ─► v (B, h, T_k, d_k)

scaled_dot_product_attention(q, k, v, mask) ─► (B, h, T_q, d_k)     ← all heads at once (batched matmul)
merge ─► (B, T_q, d_model) ─ w_o ─► (B, T_q, d_model)
```

### ✍️ Your turn: write `MultiHeadAttention` in `minitf/attention.py`
`__init__`, `split_heads`, `merge_heads` and `forward`. The check **copies your weights into PyTorch's own `nn.MultiheadAttention`** and compares the outputs, so if it passes, your implementation does exactly what the library does.

In [ ]:
check.check_mha(MultiHeadAttention)

## 3. Parameter count (a classic interview question)

### ✍️ Your turn
Write the number of parameters in one `MultiHeadAttention(d_model=512, n_heads=8)` as a formula. (Four linear layers of size d_model × d_model, each with a bias.) Does it depend on `n_heads`?

In [ ]:
mha_params = None   # TODO: a formula, not a number from running the code

In [ ]:
count = lambda m: sum(p.numel() for p in m.parameters())
check.equal("params of MHA(512, 8)", mha_params, count(MultiHeadAttention(512, 8)))
print("1 head:", count(MultiHeadAttention(512, 1)), "| 8 heads:", count(MultiHeadAttention(512, 8)),
      "| 64 heads:", count(MultiHeadAttention(512, 64)))

The same parameters and the same compute (FLOPs), whatever $h$ is. More heads means **more, smaller** attention patterns instead of one big one.

## 4. Attention has no idea about word order

### ✍️ Your turn: an experiment
Shuffle the tokens of a sequence, run self-attention, and compare with the original output.
1. `out = mha(x, x)`
2. `x_shuffled = x[:, perm]` (reorders the tokens)
3. `out_shuffled = mha(x_shuffled, x_shuffled)`

If attention ignores order, then `out_shuffled` should equal `out[:, perm]`: the same outputs, just shuffled the same way.

In [ ]:
torch.manual_seed(0);
mha = MultiHeadAttention(16, 4)
x = torch.randn(1, 5, 16)
perm = torch.tensor([3, 0, 4, 1, 2])
out = None            # TODO
out_shuffled = None   # TODO

In [ ]:
check.close("out_shuffled == out[:, perm]", out_shuffled, out[:, perm])

Attention is **permutation-equivariant**: it treats a sentence as a **set**. Each token in *"dog bites man"* gets exactly the same vector as in *"man bites dog"*.
(RNNs never had this problem, because they read tokens one at a time. That's also why they're slow.) We need to **inject position information**.

## 5. Sinusoidal positional encoding

The paper adds a fixed vector $PE_{pos}$ to the embedding of the token at position $pos$:

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \qquad PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

**Intuition: a clock with many hands.** Each pair of dimensions $(2i, 2i{+}1)$ is a hand that turns at its own speed. Early dimensions spin fast (they tell nearby positions apart), and later ones turn very slowly (they give the coarse position). Together the readings form a unique "timestamp" for each position, a smooth version of counting in binary.

### ✍️ Your turn: write `sinusoidal_positional_encoding` in `minitf/positional.py`

In [ ]:
check.check_positional(sinusoidal_positional_encoding)

In [ ]:
pe = sinusoidal_positional_encoding(100, 64)
viz.plot_matrix(pe, "positional encoding: rows = positions, columns = dimensions",
                xlabel="dimension", ylabel="position", cmap="RdBu")

**A useful property:** the similarity between two positions depends (almost) only on **how far apart** they are, and not on where they are.

In [ ]:
sim = pe @ pe.T
viz.plot_matrix(sim[:40, :40], "PE[p] · PE[q]: a band along the diagonal (depends on |p − q|)", xlabel="q", ylabel="p")
plt.figure(figsize=(7, 3))
for p in [10, 30, 50]:
    plt.plot(sim[p].numpy(), label=f"position {p}")
plt.xlabel("other position"); plt.ylabel("dot product"); plt.legend(); plt.grid(alpha=.3); plt.show()

Each curve has the same shape, just shifted. The paper's reason for this design: for any fixed offset $k$, $PE_{pos+k}$ is a **linear function** (a rotation) of $PE_{pos}$. So the model can easily learn to attend "3 tokens back" with one linear map, wherever it is in the sentence.

**Add or concatenate?** Adding keeps the size at $d_{model}$. In high dimensions the model can learn to keep "what" and "where" in mostly separate directions, so mixing them costs very little.

### Re-run the shuffle test, now with positions added

In [ ]:
pos = sinusoidal_positional_encoding(5, 16)[None]          # (1, 5, 16)
x_pos = x + pos
xs_pos = x_shuffled + pos                                   # the shuffled tokens get the positions of where they now are
out_pos, out_shuffled_pos = mha(x_pos, x_pos), mha(xs_pos, xs_pos)
print("still equivariant?", torch.allclose(out_shuffled_pos, out_pos[:, perm], atol=1e-5))

`False`: word order changes the result now. ✅

### Other ways to encode position (know these for interviews)
| method | used by | idea |
|---|---|---|
| **sinusoidal** (absolute, fixed) | original Transformer | the formula above, added to the embeddings |
| **learned absolute** | BERT, GPT-2 | an `nn.Embedding(max_len, d)` table, one learned vector per position. Can't go beyond `max_len` |
| **RoPE** (rotary) | Llama, Mistral, most modern LLMs | **rotates** $q$ and $k$ by an angle proportional to position, so $q_i\cdot k_j$ depends only on $i - j$ (relative position). Applied inside attention, not added to the embeddings |
| **ALiBi** | BLOOM | adds a penalty to the scores that grows with distance |

## ✅ Check your understanding

<details><summary>1. Why do we need W_Q, W_K, W_V at all?</summary>

Without them, $q = k = v = x$: every token mostly attends to itself, the pattern is symmetric, and nothing is learned about *what to look for*. The projections let the model learn separate "search", "advertise" and "share" representations.
</details>

<details><summary>2. Why multiple heads? Is it more expensive than one head?</summary>

Each head can learn a different relationship (syntax, coreference, position offsets...). It is **not** more expensive: $d_{model}$ is split into $h$ heads of $d_k = d_{model}/h$, so the parameters and FLOPs are the same as one full-width head.
</details>

<details><summary>3. Parameters in one multi-head attention layer?</summary>

$4(d_{model}^2 + d_{model})$, which is 1,050,624 for $d_{model} = 512$. It doesn't depend on $h$.
</details>

<details><summary>4. Walk through the shapes inside multi-head attention.</summary>

(B, T, d) → linear → (B, T, d) → view (B, T, h, d_k) → transpose (B, h, T, d_k) → scores (B, h, T, T) → out (B, h, T, d_k) → transpose + view (B, T, d) → W_O → (B, T, d).
</details>

<details><summary>5. Why does a Transformer need positional information? Does an RNN?</summary>

Self-attention is permutation-equivariant, so it sees a bag of tokens. An RNN processes tokens in order, so order is built into its computation.
</details>

<details><summary>6. Why sin/cos with different frequencies?</summary>

Each position gets a unique, bounded pattern; nearby positions get similar codes; and $PE_{pos+k}$ is a linear (rotation) function of $PE_{pos}$, which makes relative positions easy to learn. The values stay in [−1, 1] for any length.
</details>

<details><summary>7. What is RoPE in one sentence?</summary>

It rotates each query and key vector by an angle proportional to its position, so their dot product depends only on the *relative* distance $i - j$.
</details>

**Next: `04_transformer.ipynb`**